## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [11]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [ ]:
MODEL = "gemma3:1b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [13]:
retriever = vectorstore.as_retriever()
MODEL = "gemma3:1b"
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='2a480612-bdce-4dc2-a079-cacf867cca92', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [15]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery is a fascinating and incredibly complex character in the Netflix series *Stranger Things*. Here\'s a breakdown of who she is and why she\'s so captivating:\n\n**Core Identity:**\n\n* **Real Name:** Avery Cooper\n* **Age:** 19\n* **Occupation:** She\'s a talented, but somewhat troubled, musician and aspiring filmmaker. She\'s working as a "ghostwriter" for a local music scene, primarily writing songs for other artists.\n\n**The Mystery of Her Past:**\n\n* **The "Ghost" Connection:** Avery\'s primary and most significant mystery revolves around her connection to the Upside Down. She\'s a "ghost" – a being from the Upside Down who has been trapped for decades.  She\'s been observing and interacting with the Hawkins residents for a long time, and her presence is subtly influencing events.\n* **The "Ghostly" Abilities:**  Avery possesses a unique ability to manipulate the Upside Down – she can "read" the memories of those trapped within it, and she can even briefly 

## Time to put this together!

In [16]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("Who is Averi Lancaster?", [])

'Okay, let’s talk about Averi Lancaster.\n\nBased on the context provided – specifically the “Signatures” and “Summary” – Averi Lancaster appears to be a **key member of the Insurellm team as a UX Designer.**\n\nHere’s what we know so far:\n\n*   **Role:** She leads the design for the Homellm home insurance portal.\n*   **Experience:** She has a strong background in UX design, with a focus on user research and improving user satisfaction.\n*   **Background:** She’s a recent graduate from Harmony Health Plans and has been with Insurellm for a significant period.\n\n**In short, Averi Lancaster is a vital part of the Insurellm team, focusing on the user experience of their online platform.**\n\nDo you have any specific questions about Averi Lancaster that you’d like me to answer?'

## What could possibly come next? 😂

In [19]:
gr.ChatInterface(answer_question).launch()

c:\JM\LLM_course\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!